# 02c · 세분화, 중첩, 비동질 푸아송 과정 (Thinning, Superposition, and the Nonhomogeneous Poisson Process)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §2.4 (변환: 세분화·중첩·조건화 — 세분화 정리의 결합 pmf 인수분해 증명) · Ross 12e §5.3.4 (세분화 Prop 5.2, 중첩), §5.4.1 (NHPP의 무한소 정의, 시각 의존 세분화) · G&S 4e §6.8 (colouring theorem = 색이 여러 개인 세분화, 중첩, 비동질) |
| 선수 노트북 | 02b (세 정의, 조건부 균등 순서통계량), 02a (경쟁하는 지수 시계) |
| 예상 소요 | 90분 |
| 상태 | draft |

이 노트북의 단 하나의 아이디어: **푸아송 과정의 점마다 독립 동전을 던지면 색깔별로 다시 (서로 독립인) 푸아송 과정이 된다.** 동전 확률이 상수면 세분화, 색깔 있는 점을 합치면 중첩, 동전 확률이 시각에 따라 변하면 비동질 푸아송 과정(NHPP)이다. 세 변환의 증명은 모두 02b의 '조건부 균등성 + 이항 → 푸아송 인수분해' 한 줄로 환원된다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy import stats
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.poisson import poisson_arrivals, thin, nhpp_thinning
from spkit.renewal import renewal_count
from spkit.plots import plot_hist_vs_pmf, plot_hist_vs_pdf

SEED, rng = rng_for("02c")
setup_plots()
N_PATHS = scale(50_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
T = 10.0                  # 이 노트북 전체의 지평
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (02b) 푸아송 과정의 '간격 정의'를 말하라.

<details><summary>정답</summary>

iid $\tau_i\sim\mathrm{Exp}(\lambda)$, $T_n=\sum_{i\le n}\tau_i$, $N(t)=\max\{n:T_n\le t\}$. 이 정의가 시뮬레이션(`poisson_arrivals`)의 근거다.

</details>

**Q2.** (02b) $N(T)=n$이 주어졌을 때 도착시각들의 조건부 분포는?

<details><summary>정답</summary>

$n$개 iid $U(0,T)$의 순서통계량 (결합밀도 $n!/T^n$). 순서를 무시하면 iid 균등. 오늘 세 정리의 증명이 모두 이 사실에서 시작한다.

</details>

**Q3.** (02b) 길이 $h$인 작은 구간에 도착이 2개 이상일 확률의 차수는? 정확한 값은?

<details><summary>정답</summary>

$o(h)$; 정확히 $1-e^{-\lambda h}(1+\lambda h)\approx(\lambda h)^2/2$. 중첩에서 두 과정이 **독립**이어야 하는 이유가 여기 있다 (한 시각에 2개가 몰리면 이 성질이 깨진다).

</details>

**Q4 (연결 고리).** (02a) 경쟁하는 지수 시계에서 '최솟값 $T$와 승자 $I$가 독립'이라는 말의 뜻과 공식은?

<details><summary>정답</summary>

$P(I=i,\,T>t)=\frac{\lambda_i}{\Lambda}e^{-\Lambda t}=P(I=i)\,P(T>t)$ ($\Lambda=\sum_j\lambda_j$). 누가 이기든 승리 시각의 분포가 같다. 오늘 중첩 정리의 '출처 라벨은 독립 동전'이 바로 이 사실이다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **세분화(thinning).** 비율 $\lambda$ 푸아송 과정의 각 도착을 (다른 도착·과정과 독립인) 동전으로 확률 $p$로 '유지(type 1)', $1-p$로 '제거(type 2)'한다. $N_1(t)$ = 유지된 도착 수, $N_2(t)$ = 제거된 도착 수, $N=N_1+N_2$. spkit: `thin(times, p, rng) -> (kept, removed)`.
- **중첩(superposition).** 독립 푸아송 과정 $N^{(1)}$ (비율 $\lambda_1$), $N^{(2)}$ (비율 $\lambda_2$)의 합 $N=N^{(1)}+N^{(2)}$. 도착시각을 합쳐 정렬한 것이며, 각 도착에 '어느 과정에서 왔는가'라는 라벨을 붙일 수 있다.
- **비동질 푸아송 과정(nonhomogeneous Poisson process, NHPP).** 비율함수(intensity) $\lambda(t)\ge0$, 누적비율 $\Lambda(t)=\int_0^t\lambda(u)\,du$. 정의: $N(0)=0$, 독립 증분, $N(t)-N(s)\sim\mathrm{Poisson}(\Lambda(t)-\Lambda(s))$. 무한소 형태: $P(N(t+h)-N(t)=1)=\lambda(t)h+o(h)$, $P(\ge2)=o(h)$. **정상 증분은 없다.**
- **NHPP 세분화 알고리즘.** $\lambda_{\max}\ge\sup_{t\le T}\lambda(t)$를 잡고 비율 $\lambda_{\max}$ 동질 과정을 만든 뒤, 시각 $t$의 도착을 확률 $\lambda(t)/\lambda_{\max}$로 유지한다. spkit: `nhpp_thinning(rate_fn, rate_max, T, rng)`. 채택률(유지되는 후보의 비율)은 $\Lambda(T)/(\lambda_{\max}T)$.
- **이 노트북의 비율함수.** $\lambda(t)=3+2\sin(\pi t/5)$ on $[0,10]$ (주기 10, 최솟값 1, 최댓값 5). $\Lambda(t)=3t+\frac{10}{\pi}\big(1-\cos(\pi t/5)\big)$; 미분하면 $3+2\sin(\pi t/5)$로 돌아온다. $\Lambda(10)=30$ (사인 항이 한 주기에서 상쇄), $\Lambda(2.5)=7.5+10/\pi$, $\Lambda(5)=15+20/\pi$.
- **PASTA (Poisson Arrivals See Time Averages, 예고).** 시스템 상태 $X(t)$가 미래의 푸아송 도착과 독립(lack of anticipation)이면, 푸아송 도착 시점에 관측한 상태의 장기 비율 = 상태의 시간 평균 비율. 03d에서 M/M/1로 본격 다룬다.

### 2.2 정리 1 — 세분화 (Durrett 3e §2.4, Ross 12e §5.3.4 Prop 5.2, G&S 4e §6.8 colouring theorem)

> 비율 $\lambda$ 푸아송 과정을 확률 $p$로 세분화하면 $N_1$, $N_2$는 각각 비율 $p\lambda$, $(1-p)\lambda$의 푸아송 과정이고, 두 과정은 서로 **독립**이다.

**가정이 왜 필요한가**
- 동전이 도착마다 **독립**이고 과정과도 독립이어야 한다. '번갈아 유지'(짝수 번째만) 같은 규칙은 유지된 과정의 간격을 두 지수의 합 $\mathrm{Gamma}(2,\lambda)$로 만들고 $N_1$, $N_2$를 거의 완전히 종속시킨다 (E3).
- 동전 확률 $p$가 **상수**여야 동질 푸아송이 된다. 시각에 따라 $p(t)$면 NHPP가 된다(정리 3) — 이것이 오히려 유용하다.
- 원래 과정이 **푸아송**이어야 한다. 일반 재생과정을 독립 동전으로 세분화하면 유지 과정은 여전히 재생과정이지만(간격 = 기하분포 개수만큼의 원 간격 합, 블록이 서로소라 iid) 푸아송이 아니고, 유지·제거 과정은 **독립이 아니다** (긴 원 간격 하나가 두 과정을 동시에 비운다). 정리 2 쪽에서는 독립 재생과정의 중첩이 일반적으로 재생과정조차 아니다.

<details><summary>증명 (완전 증명)</summary>

**(1) 고정 시각 $t$에서의 결합 분포.** $N(t)=N_1(t)+N_2(t)$이고, 각 도착이 독립 동전으로 유지되므로 $N(t)=n$이 주어지면 $N_1(t)\sim\mathrm{Bin}(n,p)$이다. $j,k\ge0$에 대해
$$P(N_1(t)=j,\ N_2(t)=k)=P(N(t)=j+k)\binom{j+k}{j}p^j(1-p)^k
=e^{-\lambda t}\frac{(\lambda t)^{j+k}}{(j+k)!}\cdot\frac{(j+k)!}{j!\,k!}\,p^j(1-p)^k .$$
$(j+k)!$이 약분되고 $(\lambda t)^{j+k}=(p\lambda t)^j\big((1-p)\lambda t\big)^k/\big(p^j(1-p)^k\big)$, $e^{-\lambda t}=e^{-p\lambda t}e^{-(1-p)\lambda t}$로 갈라지므로
$$P(N_1(t)=j,\ N_2(t)=k)=\Big[e^{-p\lambda t}\frac{(p\lambda t)^j}{j!}\Big]\Big[e^{-(1-p)\lambda t}\frac{((1-p)\lambda t)^k}{k!}\Big].$$
결합 pmf가 $j$만의 함수와 $k$만의 함수의 **곱**이므로 $N_1(t)$와 $N_2(t)$는 독립이고, 각 인수를 보면 $N_1(t)\sim\mathrm{Poisson}(p\lambda t)$, $N_2(t)\sim\mathrm{Poisson}((1-p)\lambda t)$.

**(2) 증분.** 서로소 구간 $I_1,\dots,I_m$을 잡자. 원 과정의 증분 $N(I_1),\dots,N(I_m)$은 독립 $\mathrm{Poisson}(\lambda|I_i|)$이고(02b 정의 1), 구간 $I_i$에 떨어진 도착들의 동전은 다른 구간의 도착·동전과 독립이다. 따라서 쌍 $(N_1(I_i),N_2(I_i))_{i\le m}$은 서로 독립이고, 각 쌍에 (1)의 계산을 $t$ 대신 $|I_i|$로 반복하면 $N_1(I_i)\sim\mathrm{Poisson}(p\lambda|I_i|)$, $N_2(I_i)\sim\mathrm{Poisson}((1-p)\lambda|I_i|)$이며 쌍 안에서도 독립이다. 즉 $N_1$과 $N_2$는 각각 독립·정상 증분을 갖는 푸아송 과정이고(정의 1), 두 과정의 유한차원 분포가 모두 곱으로 갈라지므로 과정으로서도 독립이다. $\square$

**핵심 한 줄.** $(j+k)!$이 약분되고 $e^{-\lambda t}=e^{-p\lambda t}e^{-(1-p)\lambda t}$로 갈라진다. 이항 조건부 + 푸아송 주변 = 독립 푸아송 곱. 같은 계산이 색이 $r$개일 때(다항 조건부)도 그대로 통한다 (G&S colouring theorem).

</details>

### 2.3 정리 2 — 중첩 (Durrett 3e §2.4, Ross 12e §5.3.4)

> 독립 푸아송 과정 $N^{(1)},N^{(2)}$ (비율 $\lambda_1,\lambda_2$)의 합은 비율 $\lambda_1+\lambda_2$의 푸아송 과정이다. 또한 합쳐진 과정의 각 도착은 (다른 도착과 독립적으로) 확률 $\lambda_1/(\lambda_1+\lambda_2)$로 과정 1에서 온 것이다 — 즉 중첩은 세분화의 역이다.

**가정이 왜 필요한가**
- 두 과정의 **독립성**: 종속이면 (극단적으로 같은 과정을 두 번 더하면) 한 시각에 2개가 몰려 $P(\ge2\text{ in }h)=o(h)$(Q3)가 깨진다.
- 비율 합이 **유한**해야 한다 (무한히 많은 과정을 합칠 때).

**증명 스케치.** (i) *주변분포.* 독립 푸아송 변수의 합은 푸아송: pgf가 $e^{\lambda_1t(s-1)}e^{\lambda_2t(s-1)}=e^{(\lambda_1+\lambda_2)t(s-1)}$. (ii) *증분.* 서로소 구간에서 $N(I_i)=N^{(1)}(I_i)+N^{(2)}(I_i)$이고, 두 과정이 독립이며 각각 독립 증분이므로 합의 증분도 독립. 정상성은 각 항의 정상성에서. (iii) *출처 라벨.* 고정 시각 $s$, 또는 합쳐진 과정의 도착 시각(정지 시간; 강마르코프 성질) 직후 다음 도착까지, 무기억성에 의해 과정 1의 잔여 시계는 $\mathrm{Exp}(\lambda_1)$, 과정 2의 잔여 시계는 $\mathrm{Exp}(\lambda_2)$이고 둘은 독립·과거와 독립이다. 02a 정리 2(Q4)에 의해 승자는 $\lambda_1/(\lambda_1+\lambda_2)$ 동전이고 승리 시각 $\mathrm{Exp}(\lambda_1+\lambda_2)$과 독립 — 이것이 합쳐진 간격이 $\mathrm{Exp}(\lambda_1+\lambda_2)$이고 라벨이 (간격과도, 다른 라벨과도) 독립인 동전인 이유다. 다르게 말하면, 합쳐진 과정을 확률 $\lambda_1/(\lambda_1+\lambda_2)$로 세분화하면 정리 1에 의해 정확히 $(N^{(1)},N^{(2)})$의 분포가 나온다. 완전한 증명은 Durrett 3e §2.4.

### 2.4 정리 3 — 시각 의존 세분화 = NHPP (Ross 12e §5.4.1)

> 비율 $\lambda_{\max}$ 푸아송 과정의 시각 $t$ 도착을 (독립 동전으로) 확률 $q(t)=\lambda(t)/\lambda_{\max}\in[0,1]$로 유지하면, 유지된 과정은 비율함수 $\lambda(t)$의 NHPP다: 임의 구간 $(a,b]$의 유지 도착 수는 $\mathrm{Poisson}\big(\int_a^b\lambda(u)\,du\big)$이고 서로소 구간에서 독립이다.

**가정이 왜 필요한가**
- $\lambda(t)\le\lambda_{\max}$ **모든** $t$에서: 아니면 유지 확률 $\lambda(t)/\lambda_{\max}$가 1로 잘려 실제 비율이 $\min(\lambda(t),\lambda_{\max})$가 된다. 이 노트북의 예에서 $\lambda_{\max}=3$을 쓰면 $E[N(10)]=\int_0^{10}\min(\lambda,3)=30-20/\pi\approx23.6$ (4.4절, trap 카드).
- 동전이 도착 **시각에만** 의존하고 다른 도착과 독립이어야 한다 (정리 1과 같은 이유).

<details><summary>증명 (완전 증명)</summary>

구간 $(a,b]$를 고정하고 $M$ = 이 구간의 동질($\lambda_{\max}$) 후보 도착 수, $K$ = 그중 유지된 수라 하자. $M\sim\mathrm{Poisson}(\lambda_{\max}(b-a))$.

**(1) 조건부 균등성 → 같은 동전.** $M=n$이 주어지면 후보 시각 $S_1,\dots,S_n$은 (순서를 무시하면) iid $U(a,b)$ (02b 정리 2). 시각 $S_i$의 후보가 유지될 확률은, $S_i$에 대해 평균하면
$$\bar q=E\big[q(S_i)\big]=\frac{1}{b-a}\int_a^b\frac{\lambda(u)}{\lambda_{\max}}\,du ,$$
이고 $S_i$들이 독립, 동전이 독립이므로 유지 여부 $\xi_i=\mathbf 1\{U_i<q(S_i)\}$는 iid $\mathrm{Bernoulli}(\bar q)$. 따라서 $K\mid M=n\sim\mathrm{Bin}(n,\bar q)$.

**(2) 정리 1의 계산 재사용.** 이항 조건부 + 푸아송 주변이므로 정리 1의 (1)과 동일하게
$$P(K=j)=\sum_{n\ge j}e^{-\lambda_{\max}(b-a)}\frac{(\lambda_{\max}(b-a))^n}{n!}\binom nj\bar q^j(1-\bar q)^{n-j}
=e^{-\lambda_{\max}(b-a)\bar q}\frac{(\lambda_{\max}(b-a)\bar q)^j}{j!},$$
즉 $K\sim\mathrm{Poisson}\big(\lambda_{\max}(b-a)\bar q\big)=\mathrm{Poisson}\Big(\int_a^b\lambda(u)\,du\Big)=\mathrm{Poisson}(\Lambda(b)-\Lambda(a))$.

**(3) 서로소 구간의 독립성.** 서로소 구간들의 후보 수는 동질 과정의 독립 증분이고, 각 구간의 후보 시각·동전은 다른 구간과 독립이므로 (1)–(2)를 구간별로 적용하면 유지 수들이 독립이다. $N(0)=0$은 자명. 따라서 유지된 과정은 정의(2.1)의 NHPP다. $\square$

**따름정리 (조건부 도착시각).** $N(T)=n$이 주어지면 NHPP의 도착시각은 (순서 무시) iid, 밀도 $\lambda(t)/\Lambda(T)$: 후보는 균등이고 시각 $t$에서 살아남을 확률이 $\lambda(t)/\lambda_{\max}$에 비례하기 때문 (Bayes). 4.3절 (b)에서 풀링 히스토그램이 $\lambda(t)$ 모양을 보이는 이유다.

</details>

### 2.5 직관

푸아송 과정은 '점마다 아무 기억 없이 독립적으로 뿌려진 점들'이다. 그래서 점마다 독립 동전을 던져 색칠하면 색별로 다시 푸아송이고, 색들 사이에도 정보가 새지 않는다(세분화). 거꾸로 색깔 있는 점들을 합치면 색이 독립 동전이 된다(중첩). 동전 확률이 시각에 따라 변하면 '점의 밀도'가 시각에 따라 변하는 과정, 즉 NHPP가 되며, 그래서 NHPP 시뮬레이션은 '넉넉한 균일 비율로 뿌린 뒤 지역별로 솎아내기'다. 세 변환 모두 증명의 실질은 하나 — **조건부 균등성 + 이항–푸아송 인수분해**. PASTA도 같은 뿌리에서 나온다: 푸아송 도착은 시스템을 '엿보지 못하므로' 무작위 시각에 시스템을 보는 것과 같다.

### 2.6 함정

1. `nhpp_thinning`류 알고리즘의 `rate_max`가 $\sup\lambda(t)$보다 작으면 **조용히** 틀린다: 채택확률이 1로 잘려 비율이 $\min(\lambda(t),\lambda_{\max})$가 된다. 이 노트북의 예에서 $\lambda_{\max}=3$이면 $E[N(10)]$이 30이 아니라 $30-20/\pi\approx23.6$. (spkit의 `nhpp_thinning`은 이 경우 `ValueError`를 내도록 가드가 있지만, 직접 짠 코드에는 없다 — 4.4절.)
2. NHPP는 정상 증분이 없다: $N(t)-N(s)$의 분포는 $t-s$만이 아니라 $s,t$ 자체에 의존한다. 조건부 도착시각도 균등이 아니라 밀도 $\lambda(t)/\Lambda(T)$.
3. 세분화의 두 과정이 **독립**이라는 것은 '$N_1(t)$와 $N_2(t)$가 무상관'보다 훨씬 강한 주장이다(모든 유한차원 분포가 곱). 반대로 원 과정 $N=N_1+N_2$와 $N_1$은 당연히 종속이다(양의 상관, $E[N_1\mid N=n]=pn$).
4. 중첩에서 '출처 라벨이 독립 동전'인 것은 지수 간격일 때만이다. 규칙적인 두 과정을 합치면 라벨이 번갈아 나오는 경향이 생긴다.
5. PASTA는 '도착이 시스템을 미리 보지 못한다'는 가정이 필요하다. 주기적/결정론적 도착은 시간 평균을 보지 못한다 (4.5절의 대조군).
6. (시뮬레이션 함정) 유한 창 $(0,T]$ 안에서 '연속된 두 도착 사이의 간격'을 모으면 마지막 절단된 간격이 빠져 평균이 $1/\lambda$보다 **작게** 편향된다 — 창 안에 다음 도착이 있어야 하는 조건이 짧은 간격을 골라내기 때문. 4.2절에서 숫자로 본다 (02d 검사 역설의 예고).

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요) — (1) 세분화 정리에서 '독립'이 어디서 튀어나오는지 한 문장으로 (힌트: $(j+k)!$). (2) NHPP를 만들 때 왜 $\lambda_{\max}$가 필요한지, 너무 크게 잡으면 무슨 대가를 치르는지(효율 $\Lambda(T)/(\lambda_{\max}T)$) 두 문장으로.

## 3. Predict — 코드를 돌리기 전에 예측

아래 셀의 `None`을 숫자로 바꿔 보세요. 정확할 필요는 없습니다 — 자릿수와 방향이 맞는지가 목적입니다. 예측을 먼저 적어 두고 4절을 실행하세요.

In [ ]:
predictions = {
    # 비율 5, T=10 인 과정을 p=0.3 으로 세분화. 유지된 도착 수의 평균은?
    "mean_kept": None,
    # 유지 수와 제거 수의 상관계수는? (양수? 음수? 0?)
    "corr_kept_removed": None,
    # 비율 1 과 비율 3 인 독립 과정을 합치면, 합쳐진 도착 중 과정 2 에서 온 비율은?
    "frac_from_2": None,
    # lambda(t) = 3 + 2 sin(pi t / 5) 인 NHPP 의 E[N(10)] 은?
    "mean_N10_nhpp": None,
    # 같은 NHPP 의 E[N(2.5)] 는? (동질(비율 3)이라면 7.5 — 더 클까 작을까?)
    "mean_N25_nhpp": None,
    # 시스템이 '켜짐 1초, 꺼짐 2초'를 결정론적으로 반복하고 비율 5 인 푸아송 도착이
    # 시스템을 관측할 때, 켜진 상태를 보는 도착의 비율은?
    "p_see_on": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 세분화 (정리 1)

비율 5, $T=10$인 과정을 $p=0.3$으로 세분화한다. 경로마다 유지 수 $K$와 제거 수 $R$을 기록한다. 정리 1이 맞다면 $K\sim\mathrm{Poisson}(15)$, $R\sim\mathrm{Poisson}(35)$이고 둘은 **독립**이어야 한다.

In [ ]:
LAM, P_KEEP = 5.0, 0.3
K = np.empty(N_PATHS, dtype=int); R = np.empty(N_PATHS, dtype=int)
for i in range(N_PATHS):
    times = poisson_arrivals(LAM, T, rng)
    kept, removed = thin(times, P_KEEP, rng)
    K[i], R[i] = len(kept), len(removed)
    if i == 0:
        first_thin = (times, kept, removed)
N_tot = K + R

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.eventplot(first_thin, lineoffsets=[2, 1, 0], linelengths=0.8, colors=["k", "C2", "C3"])
ax1.set_yticks([2, 1, 0]); ax1.set_yticklabels(["original (rate 5)", "kept (p = 0.3)", "removed"])
ax1.set_xlabel("t"); ax1.set_title("Fig 1a: thinning one path with independent coins")
k = np.arange(0, 36)
plot_hist_vs_pmf(K, k, stats.poisson(P_KEEP * LAM * T).pmf(k), ax=ax2, label_pmf="Poisson(15) pmf")
ax2.set_xlabel("kept count K = N_1(10)"); ax2.set_ylabel("density")
ax2.set_title(f"Fig 1b: kept count over {N_PATHS} paths")
plt.tight_layout(); plt.show()

독립성을 눈으로 보자. $K=k$인 경로만 골라 $R$의 평균을 구하면 $k$에 무관하게 $(1-p)\lambda T=35$로 **평평**해야 한다(정리 1). 대조군: 원 과정 $N=K+R$에 조건부인 $K$의 평균은 $E[K\mid N=n]=pn$으로 기울기 $0.3$ — 세분화된 두 과정은 서로 독립이지만 각각은 원 과정과 종속이다 (함정 3).

In [ ]:
ks = np.arange(10, 21); ns = np.arange(40, 61)
ER_given_K = np.array([R[K == kk].mean() for kk in ks])
EK_given_N = np.array([K[N_tot == nn].mean() for nn in ns])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.bar(ks, ER_given_K, color="C3", alpha=0.6, label="E[R | K = k] (MC)")
ax1.axhline((1 - P_KEEP) * LAM * T, color="k", ls="--", label="(1-p) lambda T = 35")
ax1.set_ylim(30, 40); ax1.set_xlabel("kept count k"); ax1.set_ylabel("mean removed count"); ax1.legend()
ax1.set_title("Fig 2a: kept and removed are independent (flat)")
ax2.plot(ns, EK_given_N, "C2o", label="E[K | N = n] (MC)")
ax2.plot(ns, P_KEEP * ns, "k--", label="p n (slope 0.3)")
ax2.set_xlabel("total count n"); ax2.set_ylabel("mean kept count"); ax2.legend()
ax2.set_title("Fig 2b: ... but kept depends on the total")
plt.tight_layout(); plt.show()

est_K = mc_estimate(K); est_pK15 = mc_proportion(K == 15)
corr_KR = np.corrcoef(K, R)[0, 1]
print(f"E[K]       MC {est_K}    exact p lambda T = {P_KEEP * LAM * T}")
print(f"P(K = 15)  MC {est_pK15}    exact {stats.poisson(P_KEEP * LAM * T).pmf(15):.5f}")
print(f"corr(K, R) = {corr_KR:+.4f}   (SE ≈ 1/sqrt(n) = {1 / np.sqrt(N_PATHS):.4f});   corr(K, N) = {np.corrcoef(K, N_tot)[0, 1]:+.3f}  (exact sqrt(p) = {np.sqrt(P_KEEP):.3f})")

$\mathrm{corr}(K,N)=\sqrt p$: $\mathrm{Cov}(K,K+R)=\mathrm{Var}K=p\lambda T$, $\sqrt{\mathrm{Var}K\,\mathrm{Var}N}=\sqrt{p}\,\lambda T$.

### 4.2 중첩 (정리 2)

비율 1과 비율 3인 독립 과정을 합친다. 확인할 것: (a) 합쳐진 $N(10)\sim\mathrm{Poisson}(40)$, (b) 각 도착의 출처 라벨 = 확률 $3/4$ 동전, (c) 합쳐진 간격 $\sim\mathrm{Exp}(4)$, (d) 연속된 두 라벨의 상관 $\approx0$.

(c)에는 함정 6이 숨어 있다. 지평을 $2T=20$으로 넉넉히 잡고 **처음 40개 간격만** 쓰면 절단 편향이 사실상 0이다 ($P(N(20)\le40)=P(\mathrm{Poisson}(80)\le40)\approx10^{-6}$). 비교를 위해 창 $(0,10]$ 안의 간격 전체를 순진하게 모은 평균도 함께 출력한다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def merge_labeled(t1, t2):
    """두 도착 배열을 합쳐 정렬; (merged_sorted_times, labels 0/1) 반환."""
    t = np.concatenate([t1, t2]); lab = np.concatenate([np.zeros(len(t1), int), np.ones(len(t2), int)])
    o = np.argsort(t, kind="stable"); return t[o], lab[o]

LAM1, LAM2, N_GAPS = 1.0, 3.0, 40
M = np.empty(N_PATHS, dtype=int); labs, gaps_first, gaps_naive = [], [], []
for i in range(N_PATHS):
    merged, lab = merge_labeled(poisson_arrivals(LAM1, 2 * T, rng), poisson_arrivals(LAM2, 2 * T, rng))
    M[i] = (merged <= T).sum()
    labs.append(lab)
    gaps_first.append(np.diff(merged[:N_GAPS + 1]))      # 처음 40개 간격: 절단 없음
    gaps_naive.append(np.diff(merged[merged <= T]))      # 창 (0,10] 안의 간격 전부: 절단 편향
lab_all = np.concatenate(labs); gaps_all = np.concatenate(gaps_first)
lab_prev = np.concatenate([l[:-1] for l in labs]); lab_next = np.concatenate([l[1:] for l in labs])

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
k = np.arange(15, 70)
plot_hist_vs_pmf(M, k, stats.poisson((LAM1 + LAM2) * T).pmf(k), ax=ax1, label_pmf="Poisson(40) pmf")
ax1.set_xlabel("merged count N(10)"); ax1.set_ylabel("density"); ax1.set_title("Fig 3a: rates add: 1 + 3 = 4")
x = np.linspace(0, 2, 200)
plot_hist_vs_pdf(gaps_all, x, stats.expon(scale=1 / (LAM1 + LAM2)).pdf(x), ax=ax2, bins=60, label_pdf="Exp(4) pdf")
ax2.set_xlim(0, 2); ax2.set_xlabel("merged inter-arrival gap"); ax2.set_ylabel("density")
ax2.set_title("Fig 3b: merged gaps are Exp(4); labels are 3/4 coins")
plt.tight_layout(); plt.show()

est_M, est_lab, est_gap = mc_estimate(M), mc_proportion(lab_all == 1), mc_estimate(gaps_all)
corr_lab = np.corrcoef(lab_prev, lab_next)[0, 1]
print(f"E[N(10)]            MC {est_M}    exact (lam1+lam2) T = {(LAM1 + LAM2) * T}")
print(f"P(label = 2)        MC {est_lab}    exact lam2/(lam1+lam2) = {LAM2 / (LAM1 + LAM2)}")
print(f"E[gap] (first 40)   MC {est_gap}    exact 1/(lam1+lam2) = {1 / (LAM1 + LAM2)}")
naive = mc_estimate(np.concatenate(gaps_naive))
print(f"E[gap] (all in window, naive)  {naive}   <- biased low (trap 6): z = {(1 / (LAM1 + LAM2) - naive.mean) / naive.se:.1f} SE below 0.25")
print(f"corr(consecutive labels) = {corr_lab:+.4f}   (SE ≈ {1 / np.sqrt(lab_prev.size):.4f})")

순진한 창 안 평균은 (출력된 $z$가 보여 주듯) SE의 10배 이상 $0.25$ 아래에 있다 — 시뮬레이션이 틀린 게 아니라 **추정량**이 편향된 것이다. 절단이 없는 처음 40개 간격은 정확히 $\mathrm{Exp}(4)$다.

### 4.3 NHPP — 세분화로 만들기 (정리 3)

$\lambda(t)=3+2\sin(\pi t/5)$, $\lambda_{\max}=5$. 경로마다 `nhpp_thinning`으로 도착을 만들고 $N(10)$, $N(2.5)$, $N(5)$를 기록한다. 확인할 것: $N(10)\sim\mathrm{Poisson}(30)$ (평균 = 분산 = 30), $E[N(2.5)]=\Lambda(2.5)=7.5+10/\pi$ (초반이 러시아워라 7.5보다 크다), 풀링한 도착시각의 밀도 $\approx\lambda(t)$, 평균 계수 곡선 $\approx\Lambda(t)$, 서로소 구간 $(0,2.5]$와 $(2.5,5]$의 계수는 무상관.

In [ ]:
rate_fn = lambda t: 3 + 2 * np.sin(np.pi * t / 5)
Lam = lambda t: 3 * t + (10 / np.pi) * (1 - np.cos(np.pi * t / 5))
RATE_MAX = 5.0
t_grid = np.linspace(0, T, 201)
assert rate_fn(t_grid).max() <= RATE_MAX          # 정리 3의 첫 번째 가정: 항상 확인

N10 = np.empty(N_PATHS, dtype=int); N25 = np.empty(N_PATHS, dtype=int); N5 = np.empty(N_PATHS, dtype=int)
count_curve = np.zeros(t_grid.size); pooled = []
for i in range(N_PATHS):
    times = nhpp_thinning(rate_fn, RATE_MAX, T, rng)
    N10[i], N25[i], N5[i] = len(times), (times <= 2.5).sum(), (times <= 5.0).sum()
    count_curve += renewal_count(times, t_grid)
    pooled.append(times)
count_curve /= N_PATHS
pooled = np.concatenate(pooled)
n_bins = 50
hist, edges = np.histogram(pooled, bins=n_bins, range=(0, T))
lam_hat = hist / (N_PATHS * (T / n_bins))          # 경험적 비율함수

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))
mid = 0.5 * (edges[:-1] + edges[1:])
ax1.bar(mid, lam_hat, width=T / n_bins, alpha=0.5, label="empirical intensity")
ax1.plot(t_grid, rate_fn(t_grid), "C1", lw=2, label="lambda(t) = 3 + 2 sin(pi t/5)")
ax1.axhline(RATE_MAX, color="gray", ls=":", label="rate_max = 5")
ax1.set_xlabel("t"); ax1.set_ylabel("arrivals per unit time"); ax1.legend(fontsize=8)
ax1.set_title("NHPP by thinning: intensity")
ax2.plot(t_grid, count_curve, "C0", lw=2, label="mean N(t) (MC)")
ax2.plot(t_grid, Lam(t_grid), "k--", label="Lambda(t)")
ax2.plot(t_grid, 3 * t_grid, "gray", ls=":", label="3t (homogeneous)")
ax2.set_xlabel("t"); ax2.set_ylabel("N(t)"); ax2.legend(fontsize=8); ax2.set_title("Mean count curve")
k = np.arange(5, 60)
plot_hist_vs_pmf(N10, k, stats.poisson(Lam(T)).pmf(k), ax=ax3, label_pmf="Poisson(30) pmf")
ax3.set_xlabel("N(10)"); ax3.set_ylabel("density"); ax3.set_title("N(10) ~ Poisson(Lambda(10) = 30)")
plt.tight_layout(); plt.show()

분산의 표준오차는 spkit에 없어 여기서 정의한다: $\mathrm{SE}(s^2)\approx\sqrt{(m_4-m_2^2)/n}$ ($m_k$는 $k$차 중심 표본적률).

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def se_of_variance(x):
    """표본분산의 표준오차 sqrt((m4 - m2^2)/n) (m_k = k차 중심 표본적률)."""
    x = np.asarray(x, dtype=float); n = x.size; d = x - x.mean()
    m2 = (d**2).mean(); m4 = (d**4).mean()
    return float(np.sqrt((m4 - m2**2) / n))

est_N10, est_N25, est_pN30 = mc_estimate(N10), mc_estimate(N25), mc_proportion(N10 == 30)
var_N10, se_varN10 = N10.var(ddof=1), se_of_variance(N10)
corr_disjoint = np.corrcoef(N25, N5 - N25)[0, 1]
acceptance = Lam(T) / (RATE_MAX * T)
print(f"E[N(10)]     MC {est_N10}    exact Lambda(10) = {Lam(T):.4f}")
print(f"Var N(10)    sample {var_N10:.3f} ± {se_varN10:.3f}    exact Lambda(10) = {Lam(T):.4f}")
print(f"P(N(10)=30)  MC {est_pN30}    exact {stats.poisson(Lam(T)).pmf(30):.5f}")
print(f"E[N(2.5)]    MC {est_N25}    exact Lambda(2.5) = {Lam(2.5):.4f}   (homogeneous rate 3 would give 7.5)")
print(f"corr(N(2.5), N(5)-N(2.5)) = {corr_disjoint:+.4f}   (SE ≈ {1 / np.sqrt(N_PATHS):.4f}) -> independent increments")
print(f"acceptance rate Lambda(10)/(rate_max T) = {acceptance:.3f};  MC {N10.mean() / (RATE_MAX * T):.3f}")

### 4.4 $\lambda_{\max}$ 트랩 시연 (assert 없음, 숫자만)

spkit의 `nhpp_thinning`은 `rate_fn > rate_max`를 만나면 `ValueError`를 낸다. 하지만 세분화를 직접 짜면 이 가드가 없고, 채택확률 $\lambda(t)/\lambda_{\max}>1$은 그냥 1로 잘린다. 가드 없는 버전을 `rate_max=3`으로 돌려 무슨 일이 생기는지 본다. 기대: 경험적 비율이 3에서 잘리고 $E[N(10)]=\int_0^{10}\min(\lambda(t),3)\,dt=30-20/\pi$.

In [ ]:
def naive_nhpp_thinning(rate_fn, rate_max, T, rng):
    """가드 없는 세분화 (교육용 — 쓰지 말 것): rate_fn > rate_max 이면 채택확률이 1로 잘린다."""
    cand = poisson_arrivals(rate_max, T, rng)
    return cand[rng.random(cand.size) * rate_max < rate_fn(cand)]

RATE_BAD, n_bad = 3.0, N_PATHS // 5
try:
    nhpp_thinning(rate_fn, RATE_BAD, T, rng)
except ValueError as e:
    print(f"spkit.nhpp_thinning(rate_max={RATE_BAD}) -> ValueError: {e}")
bad_paths = [naive_nhpp_thinning(rate_fn, RATE_BAD, T, rng) for _ in range(n_bad)]
N10_bad = np.array([len(p) for p in bad_paths])
hist_bad, _ = np.histogram(np.concatenate(bad_paths), bins=n_bins, range=(0, T))
lam_hat_bad = hist_bad / (n_bad * (T / n_bins))
clipped_mean = Lam(T) - 20 / np.pi                 # ∫ min(lambda(t), 3) dt = 30 - 20/pi

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(mid, lam_hat_bad, width=T / n_bins, alpha=0.5, color="C3", label="empirical intensity, rate_max = 3 (naive)")
ax.plot(t_grid, rate_fn(t_grid), "C1", lw=2, label="true lambda(t)")
ax.plot(t_grid, np.minimum(rate_fn(t_grid), RATE_BAD), "k--", lw=1.5, label="min(lambda(t), 3)")
ax.set_xlabel("t"); ax.set_ylabel("arrivals per unit time"); ax.legend(fontsize=8)
ax.set_title("Wrong rate_max silently clips the intensity")
plt.show()
print(f"naive rate_max=3:  E[N(10)]  MC {mc_estimate(N10_bad)}    clipped exact 30 - 20/pi = {clipped_mean:.4f}    (should be 30)")

오류도 경고도 없이 $E[N(10)]$이 30에서 $23.6$으로 내려간다. 항상 `rate_max >= rate_fn(grid).max()`를 스스로 확인하라.

### 4.5 PASTA 예고

시스템이 '켜짐 1, 꺼짐 2'를 결정론적으로 반복한다(주기 3, 켜짐의 시간 비율 $1/3$). 비율 5 푸아송 도착이 시스템 상태를 관측하면 켜짐을 볼 비율은 시간 평균 $1/3$이어야 한다(PASTA). 대조군은 **주기적** 도착: 위상에 따라 항상 켜짐(1.0) 또는 항상 꺼짐(0.0)만 본다. 위상을 균등 난수로 흔든 결정론적 도착은 **평균적으로는** $1/3$이지만 한 경로 안에서는 여전히 0 아니면 1이다 — 시간 평균을 보는 것은 무작위 위상이 아니라 **도착 자체의 무작위성**이다.

In [ ]:
def on(t):
    """PASTA toy system: on for 1 time unit, off for 2, period 3 (deterministic)."""
    return (np.asarray(t) % 3.0) < 1.0

T_PASTA = scale(3000, minimum=300)
arr = poisson_arrivals(LAM, T_PASTA, rng)                    # Poisson arrivals, rate 5
est_on = mc_proportion(on(arr))
per_A = np.arange(0.5, T_PASTA, 3.0); per_B = np.arange(1.5, T_PASTA, 3.0)   # periodic arrivals, two phases
n_phase = scale(2000)
phase_paths = [np.arange(0, T_PASTA, 3.0) + rng.uniform(0, 3) for _ in range(n_phase)]
frac_phase = np.array([on(p).mean() for p in phase_paths])   # 경로마다 0 또는 1
print(f"Poisson arrivals   see 'on': {est_on}    time average = 1/3")
print(f"periodic, phase 0.5: {on(per_A).mean():.3f}    periodic, phase 1.5: {on(per_B).mean():.3f}")
print(f"random-phase periodic: per-path fractions take values {np.unique(frac_phase)};  mean over paths {frac_phase.mean():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.2))
for s in np.arange(0, 30, 3.0):
    ax.axvspan(s, s + 1, color="C2", alpha=0.25, lw=0)
ax.eventplot([arr[arr <= 30], per_A[per_A <= 30], per_B[per_B <= 30], phase_paths[0][phase_paths[0] <= 30]],
             lineoffsets=[3, 2, 1, 0], linelengths=0.7, colors=["k", "C0", "C3", "C1"])
ax.set_yticks([3, 2, 1, 0])
ax.set_yticklabels([f"Poisson (sees on: {est_on.mean:.2f})", f"periodic, phase 0.5 (sees on: {on(per_A).mean():.1f})",
                    f"periodic, phase 1.5 (sees on: {on(per_B).mean():.1f})", f"random phase (this path: {frac_phase[0]:.1f})"])
ax.set_xlim(0, 30); ax.set_xlabel("t   (green bands = system on)")
ax.set_title("Poisson arrivals see time averages; periodic arrivals do not")
plt.tight_layout(); plt.show()

요점: 푸아송 도착은 시스템과 독립이라 '무작위 시각의 스냅샷'과 같다(조건부 균등성!). 주기적 도착은 시스템과 위상이 맞물려 특정 상태만 본다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

In [ ]:
pK15_exact = stats.poisson(P_KEEP * LAM * T).pmf(15)
pN30_exact = stats.poisson(Lam(T)).pmf(30)
rows = [
    {"name": "E[K], kept count", "predicted": predictions["mean_kept"], "estimate": est_K, "exact": P_KEEP * LAM * T},
    {"name": "P(K = 15)", "predicted": None, "estimate": est_pK15, "exact": pK15_exact},
    {"name": "corr(K, R)", "predicted": predictions["corr_kept_removed"], "estimate": corr_KR, "se": 1 / np.sqrt(N_PATHS), "exact": 0.0},
    {"name": "E[N(10)], merged", "predicted": None, "estimate": est_M, "exact": (LAM1 + LAM2) * T},
    {"name": "P(label = process 2)", "predicted": predictions["frac_from_2"], "estimate": est_lab, "exact": LAM2 / (LAM1 + LAM2)},
    {"name": "E[merged gap] (first 40)", "predicted": None, "estimate": est_gap, "exact": 1 / (LAM1 + LAM2)},
    {"name": "E[N(10)], NHPP", "predicted": predictions["mean_N10_nhpp"], "estimate": est_N10, "exact": Lam(T)},
    {"name": "Var N(10), NHPP", "predicted": None, "estimate": var_N10, "se": se_varN10, "exact": Lam(T)},
    {"name": "E[N(2.5)], NHPP", "predicted": predictions["mean_N25_nhpp"], "estimate": est_N25, "exact": Lam(2.5)},
    {"name": "P(N(10) = 30), NHPP", "predicted": None, "estimate": est_pN30, "exact": pN30_exact},
    {"name": "P(Poisson arrival sees on)", "predicted": predictions["p_see_on"], "estimate": est_on, "exact": 1 / 3},
]
Markdown(compare_table(rows))

행별로 '불일치'가 뜻하는 것:
- **E[K]**: 어긋나면 `thin`의 유지 확률이나 `poisson_arrivals`의 비율이 틀린 것이다.
- **P(K = 15)**: 평균은 맞는데 pmf가 어긋나면 유지 수가 푸아송이 아닌 것 — 동전이 독립이 아니거나(E3) 원 과정이 푸아송이 아니다.
- **corr(K, R)**: 0에서 벗어나면 세분화 정리의 독립성이 깨진 것 (동전이 과정이나 다른 동전에 의존).
- **E[N(10)], merged**: 어긋나면 비율이 더해지지 않은 것 — 두 과정이 독립이 아니거나 합칠 때 중복·누락이 있다.
- **P(label = process 2)**: 어긋나면 '출처 = $\lambda_i/\sum\lambda_j$ 동전'(02a 경쟁 시계)이 틀린 것이다.
- **E[merged gap]**: 어긋나면 합쳐진 간격이 $\mathrm{Exp}(\lambda_1+\lambda_2)$가 아니거나, 절단된 창 안의 간격을 모아 편향된 추정량을 쓴 것이다 (함정 6).
- **E[N(10)], NHPP**: 어긋나면 `rate_max`가 $\sup\lambda$보다 작거나(4.4절) 채택확률이 $\lambda(t)/\lambda_{\max}$가 아닌 것이다.
- **Var N(10), NHPP**: 평균은 맞는데 분산이 다르면 유지된 과정이 푸아송이 아닌 것 — 예컨대 동전이 시각이 아닌 다른 도착에 의존.
- **E[N(2.5)], NHPP**: 어긋나면 $\Lambda$가 $\lambda$의 적분이 아니거나 정상 증분을 잘못 가정한 것 ($3\cdot2.5=7.5$면 정상 증분 가정의 오류).
- **P(N(10) = 30), NHPP**: 어긋나면 정리 3의 이항–푸아송 인수분해가 깨진 것이다.
- **P(sees on)**: 1/3에서 벗어나면 도착이 시스템 상태와 독립이 아닌 것 (PASTA의 가정 위반; 주기적 도착이 그 예).

In [ ]:
assert_close(est_K, P_KEEP * LAM * T, k=4, name="mean_kept")
assert_close(est_pK15, pK15_exact, k=4, name="p_kept_eq_15")
assert_close(corr_KR, 0.0, se=1 / np.sqrt(N_PATHS), k=4, name="corr_kept_removed")
assert_close(est_M, (LAM1 + LAM2) * T, k=4, name="mean_merged")
assert_close(est_lab, LAM2 / (LAM1 + LAM2), k=4, name="frac_from_2")
assert_close(est_gap, 1 / (LAM1 + LAM2), k=4, name="mean_merged_gap")
assert_close(est_N10, Lam(T), k=4, name="mean_N10_nhpp")
assert_close(var_N10, Lam(T), se=se_varN10, k=4, name="var_N10_nhpp")
assert_close(est_N25, Lam(2.5), k=4, name="mean_N25_nhpp")
assert_close(est_pN30, pN30_exact, k=4, name="p_N10_eq_30")
assert_close(est_on, 1 / 3, k=4, name="p_see_on")
print("all checks passed")

## 6. 연습문제

### E1 계산

$\lambda(t)=3+2\sin(\pi t/5)$인 NHPP. 손으로 구한 뒤 코드로 확인하라.
(a) $\Lambda(5)$를 구하라.
(b) $P(N(1)=0)$은?
(c) $P(N(5)-N(2.5)=10)$은?
(d) 이 과정을 세분화로 만들 때 $\lambda_{\max}=5$면 채택률(유지되는 후보의 비율)은 얼마인가? $\lambda_{\max}=10$이면?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $\Lambda(5)=15+\frac{10}{\pi}(1-\cos\pi)=15+\frac{20}{\pi}\approx21.3662$.
(b) $N(1)\sim\mathrm{Poisson}(\Lambda(1))$, $\Lambda(1)=3+\frac{10}{\pi}(1-\cos\frac{\pi}{5})\approx3.6079$이므로 $P(N(1)=0)=e^{-\Lambda(1)}\approx0.02711$.
(c) 증분 $N(5)-N(2.5)\sim\mathrm{Poisson}(\Lambda(5)-\Lambda(2.5))$, $\Lambda(5)-\Lambda(2.5)=7.5+\frac{10}{\pi}\big[(1-\cos\pi)-(1-\cos\tfrac{\pi}{2})\big]=7.5+\frac{10}{\pi}(2-1)=7.5+\frac{10}{\pi}\approx10.6831$ (사인이 $t=2.5$에 대해 대칭이라 $\Lambda(2.5)$와 같다). pmf at 10 $\approx0.1223$. **주의:** 정상 증분이 없으므로 $\Lambda(2.5)$를 그대로 쓸 수 있는 것은 이 대칭성 덕분이지 일반 법칙이 아니다.
(d) 채택률 $=\Lambda(10)/(\lambda_{\max}T)=30/50=0.6$; $\lambda_{\max}=10$이면 $30/100=0.3$ — 후보를 두 배 만들고 70%를 버린다. $\lambda_{\max}$는 $\sup\lambda$ 이상이되 가능한 한 작게.

```python
a = Lam(5.0)
b = np.exp(-Lam(1.0))
c = stats.poisson(Lam(5.0) - Lam(2.5)).pmf(10)
d5, d10 = Lam(10.0) / (5 * 10), Lam(10.0) / (10 * 10)
print(f"(a) {a:.4f}  (b) {b:.6f}  (c) {c:.6f}  (d) {d5:.2f}, {d10:.2f}")
# (a) 21.3662  (b) 0.027108  (c) 0.122348  (d) 0.60, 0.30
```

</details>

### E2 유도 후 시뮬레이션 검증

중첩($\lambda_1=1$, $\lambda_2=3$)에서 '과정 2의 연속된 두 도착 사이에 과정 1의 도착이 $k$개일 확률'이 기하분포
$$P(K=k)=\frac{\lambda_2}{\lambda_1+\lambda_2}\Big(\frac{\lambda_1}{\lambda_1+\lambda_2}\Big)^k,\qquad k=0,1,2,\dots$$
임을 02a의 경쟁 시계로 유도하고, $E[K]=\lambda_1/\lambda_2=1/3$과 $P(K=0)=3/4$을 시뮬레이션으로 검증하라. 4.2절의 `labs`(경로별 라벨 배열, 지평 20)를 써도 된다. 함정 6에 주의: 창 안의 모든 런을 다 쓰면 편향이 생기므로 경로마다 **처음 10개 런만** 써라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**유도.** 과정 2의 도착 직후, 무기억성으로 두 시계 $\mathrm{Exp}(\lambda_1)$, $\mathrm{Exp}(\lambda_2)$가 새로 시작한다. 다음 도착이 과정 1일 확률은 $\lambda_1/\Lambda$, 과정 2일 확률은 $\lambda_2/\Lambda$ ($\Lambda=\lambda_1+\lambda_2$)이고, 그 도착 직후 다시 두 시계가 새로 시작하므로 매 도착의 출처는 **독립 동전**이다(02a 정리 2: 승자는 승리 시각과 독립). 따라서 과정 2가 나올 때까지 과정 1이 나온 횟수 $K$는 성공확률 $q=\lambda_2/\Lambda=3/4$인 기하분포(실패 횟수): $P(K=k)=q(1-q)^k$. $E[K]=(1-q)/q=\lambda_1/\lambda_2=1/3$, $P(K=0)=q=3/4$.

**왜 처음 10개 런만 쓰는가.** 창 $(0,20]$ 안에서 라벨 열의 길이 $M$이 유한하므로 '다음 과정-2 도착이 창 안에 있어야 한다'는 조건이 뒤쪽 런을 짧게 편향시킨다 (4.2절의 간격 평균과 같은 이유). 지평 20에서 과정 2의 도착은 평균 60개이므로 처음 10개 런은 사실상 항상 존재하고 편향이 없다.

```python
K_runs = []
for lab in labs:                                   # 4.2절: 지평 20, 라벨 0/1
    idx2 = np.flatnonzero(lab == 1)                # 과정 2 도착의 위치
    K_runs.append((np.diff(idx2) - 1)[:10])        # 연속된 과정-2 도착 사이의 과정-1 개수, 처음 10개 런
K_all = np.concatenate(K_runs)
q = LAM2 / (LAM1 + LAM2)
est_EK, est_K0 = mc_estimate(K_all), mc_proportion(K_all == 0)
print(f"E[K]   MC {est_EK}   exact lam1/lam2 = {LAM1 / LAM2:.4f}")
print(f"P(K=0) MC {est_K0}   exact lam2/(lam1+lam2) = {q}")
assert_close(est_EK, LAM1 / LAM2, k=4, name="E2 mean")
assert_close(est_K0, q, k=4, name="E2 P(K=0)")
kk = np.arange(0, 8)
plot_hist_vs_pmf(K_all, kk, q * (1 - q) ** kk, label_pmf="geometric pmf, q = 3/4")
plt.xlabel("process-1 arrivals between consecutive process-2 arrivals"); plt.show()
# E[K] ≈ 0.333, P(K=0) ≈ 0.750 (FAST: SE ≈ 0.003 / 0.002)
```

</details>

### E3 가정을 깨보기

세분화 동전을 독립 동전 대신 **'번갈아 유지'**(짝수 번째 도착만 유지, `times[::2]`; 제거는 `times[1::2]`)로 바꿔라 (비율 5, $T=10$, 유지 비율은 여전히 $1/2$).
(a) 유지 수 $K'$의 평균과 분산, dispersion index(분산/평균)는? 푸아송이면 1이다.
(b) 유지 수와 제거 수의 상관계수는?
(c) 유지 과정의 간격 히스토그램을 $\mathrm{Exp}(2.5)$ 밀도와 비교하라 — 무엇으로 보이는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $K'=\lceil N/2\rceil$, $N\sim\mathrm{Poisson}(50)$이므로 평균 $\approx25$ (정확히는 $25+P(N\text{ 홀수})/2\approx25.25$), 분산 $\approx\mathrm{Var}(N)/4=12.5$, dispersion $\approx0.5$. 푸아송(1)이 아니다 — 유지 수가 원 과정보다 **규칙적**이다.
(b) $K'-R'\in\{0,1\}$이므로 상관계수 $\approx0.99$: 독립은커녕 거의 완전 종속이다.
(c) 유지 과정의 간격은 원 간격 두 개의 합 $\to\mathrm{Gamma}(2,\text{rate }5)$, 밀도 $25xe^{-5x}$. 평균은 $2/5=0.4=1/2.5$로 $\mathrm{Exp}(2.5)$와 같지만 0 근처가 비어 있고(밀도가 0에서 시작) 분산이 절반이다. **교훈:** 세분화 정리의 '푸아송'과 '독립'은 둘 다 **동전의 독립**에서 나온다. 동전을 결정론적 규칙으로 바꾸면 유지 과정은 재생과정(Erlang 간격)이지 푸아송이 아니고, 두 부분은 종속이다.

```python
n_e3 = scale(20_000)
Kp = np.empty(n_e3, dtype=int); Rp = np.empty(n_e3, dtype=int); gaps_alt = []
for i in range(n_e3):
    times = poisson_arrivals(LAM, T, rng)
    kept, removed = times[::2], times[1::2]          # 번갈아 유지: 동전이 독립이 아니다
    Kp[i], Rp[i] = len(kept), len(removed)
    gaps_alt.append(np.diff(kept[:11]))              # 처음 10개 간격 (절단 편향 없음, 함정 6)
gaps_alt = np.concatenate(gaps_alt)
print(f"(a) E[K'] {Kp.mean():.3f}   Var {Kp.var(ddof=1):.3f}   dispersion {Kp.var(ddof=1) / Kp.mean():.3f}   (Poisson: 1)")
print(f"(b) corr(K', R') = {np.corrcoef(Kp, Rp)[0, 1]:+.4f}   (independent coins gave ~0)")
x = np.linspace(0, 2.5, 200)
ax = plot_hist_vs_pdf(gaps_alt, x, stats.gamma(a=2, scale=1 / LAM).pdf(x), bins=60, label_pdf="Gamma(2, rate 5) pdf")
ax.plot(x, stats.expon(scale=1 / 2.5).pdf(x), "k--", label="Exp(2.5) pdf (same mean)"); ax.legend()
ax.set_xlabel("gap between kept arrivals (every other arrival)"); ax.set_title("Alternate keeping is not thinning")
plt.show()
# (a) ≈ 25.25 / 12.5 / 0.50   (b) ≈ +0.99   (c) Gamma(2, rate 5): zero density at 0, not Exp(2.5)
```

</details>

## 7. 정리

1. 세분화: 독립 동전으로 나눈 푸아송 과정은 비율 $p\lambda$, $(1-p)\lambda$의 **독립** 푸아송 과정 — 증명은 $(j+k)!$ 약분과 $e^{-\lambda t}$ 인수분해 한 줄.
2. 중첩: 독립 푸아송 과정의 합은 비율 합의 푸아송 과정이고, 각 도착의 출처는 $\lambda_i/\sum\lambda_j$ 독립 동전 (02a 경쟁 시계). 세분화의 역이다.
3. NHPP: $N(t)-N(s)\sim\mathrm{Poisson}(\Lambda(t)-\Lambda(s))$, $\Lambda=\int\lambda$; 정상 증분은 없다. 시뮬레이션은 비율 $\lambda_{\max}$로 뿌리고 $\lambda(t)/\lambda_{\max}$로 솎아내기 — $\lambda_{\max}\ge\sup\lambda$가 필수, 채택률은 $\Lambda(T)/(\lambda_{\max}T)$.
4. 세 변환의 증명은 모두 '조건부 균등성 + 이항 → 푸아송 인수분해' 하나로 환원된다. 유한 창 안의 간격·런을 모으면 절단 편향이 생기니 추정량을 조심하라.
5. PASTA 예고: 시스템과 독립인 푸아송 도착은 무작위 시각의 스냅샷과 같아 시간 평균을 본다; 주기적 도착은 그렇지 않다 (03d에서 M/M/1로).

**trap 카드.**
Q: 직접 짠 세분화 코드에 `rate_max=3.0`을 주고 $\lambda(t)=3+2\sin(\pi t/5)$를 넣었다. 오류 없이 돌아갔다면 결과는 맞는가?
A: 아니다. $\lambda(t)>3$인 구간에서 채택확률 $\lambda(t)/3>1$이 1로 잘려 실제 비율이 $\min(\lambda(t),3)$이 된다. $E[N(10)]=30-20/\pi\approx23.6$으로 조용히 틀린다. 항상 `rate_max >= rate_fn(grid).max()`를 assert하라 (spkit의 `nhpp_thinning`은 이 검사를 내장해 `ValueError`를 낸다).

**다음 노트북: 02d (복합 푸아송 과정과 검사 역설).** 도착마다 크기(보상, 청구액)를 붙이면 복합 푸아송 과정이 되고(00c의 랜덤 합), 고정 시각 $t$를 덮는 간격을 보면 검사 역설이 나온다. 02d는 이 두 가지로 모듈 02를 마무리하고, '무작위 시각에 관측'이라는 이 노트북의 PASTA 관점과 함정 6의 절단 편향을 길이 편향으로 다시 본다.

log/progress.md 한 줄 템플릿: `- [ ] 02c 완료 (YYYY-MM-DD) · Predict 적중 __/6 · E1 __ E2 __ E3 __ · 막힌 곳: __`